In [30]:
suppressPackageStartupMessages(library(SingleCellExperiment))
suppressPackageStartupMessages(library(scater))
suppressPackageStartupMessages(library(scran))
suppressPackageStartupMessages(library(argparse))
#####################
## Define settings ##
#####################
here::i_am("processing/1_create_seurat_rna.R")
source(here::here("settings.R"))
source(here::here("utils.R"))



here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/06_Runx1_RNA/code



In [63]:
args = list()
args$samples <- opts$samples
args$sce <- io$rna.sce
args$metadata <- "/rds/project/rds-SDzz0CATGms/users/bt392/06_Runx1_RNA/results/rna/mapping/sample_metadata_after_mapping.txt.gz"
# args$metadata <- paste0(io$basedir,"/results/rna/doublets/sample_metadata_after_doublets.txt.gz")
args$features <- 1000
args$npcs <- 20
args$test <- FALSE
args$colour_by <- c("celltype.mapped_mnn","sample")
args$vars.to.regress <- c("nFeature_RNA","mitochondrial_percent_RNA")
args$batch_correction <- c("sample")
args$remove_ExE_cells <- FALSE
args$outdir <- paste0(io$basedir,"/results/rna/dimensionality_reduction/test")

In [33]:
dir.create(args$outdir, recursive=TRUE, showWarnings = FALSE)

In [64]:
sample_metadata <- fread(args$metadata) %>%
   .[pass_rnaQC==TRUE & doublet_call==FALSE & sample%in%args$samples]

if (args$remove_ExE_cells) {
  print("Removing ExE cells...")
  sample_metadata <- sample_metadata %>%
    .[!celltype.mapped_mnn%in%c("Visceral_endoderm","ExE_endoderm","ExE_ectoderm","Parietal_endoderm")]
}

In [65]:
head(sample_metadata, 2)

cell,barcode,sample,nFeature_RNA,nCount_RNA,mitochondrial_percent_RNA,ribosomal_percent_RNA,stage,pass_rnaQC,doublet_score,doublet_call,celltype.mapped_mnn,celltype.score_mnn,closest.cell_mnn
<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<chr>,<lgl>,<dbl>,<lgl>,<chr>,<dbl>,<chr>
SLX-21184_SITTA7_H5NKNDMXY#AAACGAACAATTGCAC-1,AAACGAACAATTGCAC-1,SLX-21184_SITTA7_H5NKNDMXY,5488,57471,2.47,28.42,SLX-21184,TRUE,0.26,FALSE,Erythroid3,1.0,cell_40571
SLX-21184_SITTA7_H5NKNDMXY#AAACGAACACCAGCTG-1,AAACGAACACCAGCTG-1,SLX-21184_SITTA7_H5NKNDMXY,6686,110322,1.80,30.12,SLX-21184,TRUE,0.33,FALSE,Erythroid2,0.8,cell_64311


In [66]:
###################
## Sanity checks ##
###################

stopifnot(args$colour_by %in% colnames(sample_metadata))
# stopifnot(unique(sample_metadata$celltype.mapped) %in% names(opts$celltype.colors))

 if (length(args$batch_correction)>0) {
   stopifnot(args$batch_correction%in%colnames(sample_metadata))
   if (length(unique(sample_metadata[[args$batch_correction]]))==1) {
     message(sprintf("There is a single level for %s, no batch correction applied",args$batch_correction))
     args$batch_correction <- NULL
   } else {
     library(batchelor)
   }
 }

 if (length(args$vars_to_regress)>0) {
  stopifnot(args$vars_to_regress%in%colnames(sample_metadata))
 }

In [67]:

###############
## Load data ##
###############

# Load RNA expression data as SingleCellExperiment object
sce <- load_SingleCellExperiment(args$sce, cells=sample_metadata$cell, normalise = TRUE)

# Add sample metadata as colData
colData(sce) <- sample_metadata %>% tibble::column_to_rownames("cell") %>% DataFrame


In [71]:
head(colData(sce)[[args$batch_correction]])

[1] "SLX-21184_SITTA7_H5NKNDMXY" "SLX-21184_SITTA7_H5NKNDMXY"
[3] "SLX-21184_SITTA7_H5NKNDMXY" "SLX-21184_SITTA7_H5NKNDMXY"
[5] "SLX-21184_SITTA7_H5NKNDMXY" "SLX-21184_SITTA7_H5NKNDMXY"

In [38]:

#######################
## Feature selection ##
#######################

 if (length(args$batch_correction)>0) {
     print('batch correcting')
   decomp <- modelGeneVar(sce, block=colData(sce)[[args$batch_correction]])
 } else {
   decomp <- modelGeneVar(sce)
 }
decomp <- decomp[decomp$mean > 0.01,]
hvgs <- decomp[order(decomp$FDR),] %>% head(n=args$features) %>% rownames

# Subset SingleCellExperiment
sce_filt <- sce[hvgs,]

[1] "batch correcting"


Warning message in regularize.values(x, y, ties, missing(ties), na.rm = na.rm):
“collapsing to unique 'x' values”


In [39]:
############################
## Regress out covariates ##
############################

 if (length(args$vars_to_regress)>0) {
   print(sprintf("Regressing out variables: %s", paste(args$vars_to_regress,collapse=" ")))
   logcounts(sce_filt) <- RegressOutMatrix(
     mtx = logcounts(sce_filt), 
     covariates = colData(sce_filt)[,args$vars_to_regress,drop=F]
   )
 }


In [40]:

############################
## PCA + Batch correction ##
############################

 if (length(args$batch_correction)>0) {
   print(sprintf("Applying MNN batch correction for variable: %s", args$batch_correction))
   outfile <- sprintf("%s/pca_features%d_pcs%d_batchcorrectionby%s.txt.gz",args$outdir, args$features, args$npcs,paste(args$batch_correction,collapse="-"))
   pca <- multiBatchPCA(sce_filt, batch = colData(sce_filt)[[args$batch_correction]], d = args$npcs)
   pca.corrected <- reducedMNN(pca)$corrected
   colnames(pca.corrected) <- paste0("PC",1:ncol(pca.corrected))
   reducedDim(sce_filt, "PCA") <- pca.corrected
 } else {
   outfile <- sprintf("%s/pca_features%d_pcs%d.txt.gz",args$outdir, args$features, args$npcs)
   sce_filt <- runPCA(sce_filt, ncomponents = args$npcs, ntop=args$features)  
 }


[1] "Applying MNN batch correction for variable: sample"


In [41]:
# Save PCA coordinates
 pca.dt <- reducedDim(sce_filt,"PCA") %>% round(3) %>% as.data.table(keep.rownames = T) %>% setnames("rn","cell")
 fwrite(pca.dt, outfile)

In [44]:
args$n_neighbors = 15
args$min_dist = 0.4

In [45]:
##########
## UMAP ##
##########

# Run
set.seed(args$seed)
sce_filt <- runUMAP(sce_filt, dimred="PCA", n_neighbors = args$n_neighbors, min_dist = args$min_dist)

# Fetch UMAP coordinates
umap.dt <- reducedDim(sce_filt,"UMAP") %>% as.data.table %>% 
  .[,cell:=colnames(sce_filt)] %>%
  setnames(c("UMAP1","UMAP2","cell"))

In [46]:
# Save UMAP coordinates
fwrite(umap.dt, sprintf("%s/umap_features%d_pcs%d_neigh%d_dist%s.txt.gz",args$outdir, args$features, args$npcs, args$n_neighbors, args$min_dist))


In [47]:
##########
## Plot ##
##########

for (i in args$colour_by) {

  to.plot <- reducedDim(sce_filt,"UMAP") %>% as.data.table %>% 
    .[,cell:=colnames(sce_filt)] %>%
    merge(sample_metadata, by="cell")

  p <- ggplot(to.plot, aes_string(x="V1", y="V2", fill=i)) +
    geom_point(size=1.5, shape=21, stroke=0.05, alpha=0.5) +
    theme_classic() +
    theme(
      axis.title = element_blank(),
      axis.text = element_blank(),
      axis.ticks = element_blank()
    )

  if (grepl("celltype",i)) {
    p <- p + scale_fill_manual(values=opts$celltype.colors) +
      theme(
        legend.position="none",
        legend.title=element_blank()
      )
  }

  # Save UMAP plot
  outfile <- file.path(args$outdir,sprintf("umap_features%d_pcs%d_neigh%d_dist%s_%s.pdf",args$features, args$npcs, args$n_neighbors, args$min_dist, i))
  pdf(outfile, width=7, height=5)
  print(p)
  dev.off()
}



In [30]:
suppressPackageStartupMessages(library(SingleCellExperiment))
suppressPackageStartupMessages(library(scater))
suppressPackageStartupMessages(library(scran))
suppressPackageStartupMessages(library(argparse))
#####################
## Define settings ##
#####################
here::i_am("processing/1_create_seurat_rna.R")
source(here::here("settings.R"))
source(here::here("utils.R"))



here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/06_Runx1_RNA/code



In [48]:
args = list()
args$samples <- opts$samples
args$sce <- io$rna.sce
args$metadata <- "/rds/project/rds-SDzz0CATGms/users/bt392/06_Runx1_RNA/results/rna/mapping/sample_metadata_after_mapping.txt.gz"
# args$metadata <- paste0(io$basedir,"/results/rna/doublets/sample_metadata_after_doublets.txt.gz")
args$features <- 1000
args$npcs <- 20
args$test <- FALSE
args$colour_by <- c("celltype.mapped_mnn","sample")
args$vars.to.regress <- c("nFeature_RNA","mitochondrial_percent_RNA")
args$batch_correction <- NULL
args$remove_ExE_cells <- FALSE
args$outdir <- paste0(io$basedir,"/results/rna/dimensionality_reduction/test")

In [49]:
dir.create(args$outdir, recursive=TRUE, showWarnings = FALSE)

In [50]:
sample_metadata <- fread(args$metadata) %>%
   .[pass_rnaQC==TRUE & doublet_call==FALSE & sample%in%args$samples]

if (args$remove_ExE_cells) {
  print("Removing ExE cells...")
  sample_metadata <- sample_metadata %>%
    .[!celltype.mapped_mnn%in%c("Visceral_endoderm","ExE_endoderm","ExE_ectoderm","Parietal_endoderm")]
}

In [51]:
head(sample_metadata, 2)

cell,barcode,sample,nFeature_RNA,nCount_RNA,mitochondrial_percent_RNA,ribosomal_percent_RNA,stage,pass_rnaQC,doublet_score,doublet_call,celltype.mapped_mnn,celltype.score_mnn,closest.cell_mnn
<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<chr>,<lgl>,<dbl>,<lgl>,<chr>,<dbl>,<chr>
SLX-21184_SITTA7_H5NKNDMXY#AAACGAACAATTGCAC-1,AAACGAACAATTGCAC-1,SLX-21184_SITTA7_H5NKNDMXY,5488,57471,2.47,28.42,SLX-21184,TRUE,0.26,FALSE,Erythroid3,1.0,cell_40571
SLX-21184_SITTA7_H5NKNDMXY#AAACGAACACCAGCTG-1,AAACGAACACCAGCTG-1,SLX-21184_SITTA7_H5NKNDMXY,6686,110322,1.80,30.12,SLX-21184,TRUE,0.33,FALSE,Erythroid2,0.8,cell_64311


In [52]:
###################
## Sanity checks ##
###################

stopifnot(args$colour_by %in% colnames(sample_metadata))
# stopifnot(unique(sample_metadata$celltype.mapped) %in% names(opts$celltype.colors))

 if (length(args$batch_correction)>0) {
   stopifnot(args$batch_correction%in%colnames(sample_metadata))
   if (length(unique(sample_metadata[[args$batch_correction]]))==1) {
     message(sprintf("There is a single level for %s, no batch correction applied",args$batch_correction))
     args$batch_correction <- NULL
   } else {
     library(batchelor)
   }
 }

 if (length(args$vars_to_regress)>0) {
  stopifnot(args$vars_to_regress%in%colnames(sample_metadata))
 }

In [53]:

###############
## Load data ##
###############

# Load RNA expression data as SingleCellExperiment object
sce <- load_SingleCellExperiment(args$sce, cells=sample_metadata$cell, normalise = TRUE)

# Add sample metadata as colData
colData(sce) <- sample_metadata %>% tibble::column_to_rownames("cell") %>% DataFrame


In [54]:

#######################
## Feature selection ##
#######################

 if (length(args$batch_correction)>0) {
     print('batch correcting')
   decomp <- modelGeneVar(sce, block=colData(sce)[[args$batch_correction]])
 } else {
   decomp <- modelGeneVar(sce)
 }
decomp <- decomp[decomp$mean > 0.01,]
hvgs <- decomp[order(decomp$FDR),] %>% head(n=args$features) %>% rownames

# Subset SingleCellExperiment
sce_filt <- sce[hvgs,]

In [55]:
############################
## Regress out covariates ##
############################

 if (length(args$vars_to_regress)>0) {
   print(sprintf("Regressing out variables: %s", paste(args$vars_to_regress,collapse=" ")))
   logcounts(sce_filt) <- RegressOutMatrix(
     mtx = logcounts(sce_filt), 
     covariates = colData(sce_filt)[,args$vars_to_regress,drop=F]
   )
 }


In [56]:

############################
## PCA + Batch correction ##
############################

 if (length(args$batch_correction)>0) {
   print(sprintf("Applying MNN batch correction for variable: %s", args$batch_correction))
   outfile <- sprintf("%s/pca_features%d_pcs%d_batchcorrectionby%s.txt.gz",args$outdir, args$features, args$npcs,paste(args$batch_correction,collapse="-"))
   pca <- multiBatchPCA(sce_filt, batch = colData(sce_filt)[[args$batch_correction]], d = args$npcs)
   pca.corrected <- reducedMNN(pca)$corrected
   colnames(pca.corrected) <- paste0("PC",1:ncol(pca.corrected))
   reducedDim(sce_filt, "PCA") <- pca.corrected
 } else {
   outfile <- sprintf("%s/pca_features%d_pcs%d.txt.gz",args$outdir, args$features, args$npcs)
   sce_filt <- runPCA(sce_filt, ncomponents = args$npcs, ntop=args$features)  
 }


In [57]:
# Save PCA coordinates
 pca.dt <- reducedDim(sce_filt,"PCA") %>% round(3) %>% as.data.table(keep.rownames = T) %>% setnames("rn","cell")
 fwrite(pca.dt, outfile)

In [58]:
args$n_neighbors = 15
args$min_dist = 0.4

In [59]:
##########
## UMAP ##
##########

# Run
set.seed(args$seed)
sce_filt <- runUMAP(sce_filt, dimred="PCA", n_neighbors = args$n_neighbors, min_dist = args$min_dist)

# Fetch UMAP coordinates
umap.dt <- reducedDim(sce_filt,"UMAP") %>% as.data.table %>% 
  .[,cell:=colnames(sce_filt)] %>%
  setnames(c("UMAP1","UMAP2","cell"))

In [60]:
# Save UMAP coordinates
fwrite(umap.dt, sprintf("%s/umap_features%d_pcs%d_neigh%d_dist%s.txt.gz",args$outdir, args$features, args$npcs, args$n_neighbors, args$min_dist))


In [61]:
##########
## Plot ##
##########

for (i in args$colour_by) {

  to.plot <- reducedDim(sce_filt,"UMAP") %>% as.data.table %>% 
    .[,cell:=colnames(sce_filt)] %>%
    merge(sample_metadata, by="cell")

  p <- ggplot(to.plot, aes_string(x="V1", y="V2", fill=i)) +
    geom_point(size=1.5, shape=21, stroke=0.05, alpha=0.5) +
    theme_classic() +
    theme(
      axis.title = element_blank(),
      axis.text = element_blank(),
      axis.ticks = element_blank()
    )

  if (grepl("celltype",i)) {
    p <- p + scale_fill_manual(values=opts$celltype.colors) +
      theme(
        legend.position="none",
        legend.title=element_blank()
      )
  }

  # Save UMAP plot
  outfile <- file.path(args$outdir,sprintf("umap_features%d_pcs%d_neigh%d_dist%s_%s.pdf",args$features, args$npcs, args$n_neighbors, args$min_dist, i))
  pdf(outfile, width=7, height=5)
  print(p)
  dev.off()
}

